#### ADMFM8000 Example

In [ ]:
from adi.admfm8000 import admfm8000
from adi.ad9910 import ad9910
from adi.adf4151x import adf41513

import numpy as np

fmcw = admfm8000(uri="ip:10.32.22.147")

In [ ]:
# set DSA attenuation to 8 dB
fmcw.attenuation = 8

##### Single Tone

In [ ]:
# Single Tone Profile 0 at 25.3 GHz
fmcw.single_tone_config(
    profile=0,
    frequency=25.3e9,
    scale=1.0
)

##### Parallel Port

In [ ]:
# Parallel Port: Ramp-up Chirp
#   - Push frequencies into DMA buffer in cyclic mode and start transmission
length = 1000

freqs = np.linspace(23.8e9, 26.8e9, length)
fmcw.parallel_port_config(
    enable=True,
    frequency_np=freqs,
    cyclic=True,
    rate=(length / 100e-6) # 100 us sweep time
)

In [ ]:
# Stop Parallel Port
fmcw.parallel_port_config(enable=False)

In [ ]:
# Parallel Port: Custom pattern
#   - Push frequencies into DMA buffer in cyclic mode and start transmission
length = 1000

x = np.linspace(0, 2*np.pi, length, endpoint=False)
freqs = 1.5e9 * np.sin(x) + 25.3e9
fmcw.parallel_port_config(
    enable=True,
    frequency_np=freqs,
    cyclic=True,
    rate=(length / 100e-6) # 100 us sweep time
)

In [ ]:
# Stop Parallel Port
fmcw.parallel_port_config(enable=False)

##### Digital Ramp Generator

In [ ]:
# DRG Bidirectional continuous mode (no-dwell both high and low)
#
# 26.8 GHz __      .          .
#                 / \        / \
#                /   \      /   \
#               /     \    /     \
#              /       \  /       \
# 23.8 GHz __ /         \/         \
#             <--100us-->
#
fmcw.digital_ramp_config(
    enable=True,
    mode=ad9910.digital_ramp_generator.mode.BIDIRECTIONAL_CONTINUOUS,
    freq_min=23.8e9,
    freq_max=26.8e9,
    inc_ramp_time=50e-6,
    dec_ramp_time=50e-6,
)

In [ ]:
# DRG Ramp up mode (no-dwell high)
#   - Requires IIO backend support for DRCTL toggling
#
# 26.8 GHz __      .          .
#                 /|         /|
#                / |        / |
#               /  |       /  |
#              /   |      /   |
# 23.8 GHz __ /    |_____/    |_____
#             <--100us-->
#
fmcw.digital_ramp_config(
    enable=True,
    mode=ad9910.digital_ramp_generator.mode.RAMP_UP,
    freq_min=23.8e9,
    freq_max=26.8e9,
    inc_ramp_time=50e-6,
    toggle_en=1,            # enable auto DRCTL toggling
    ramp_delay=50e-6,       # 50 us delay between ramps
)

In [ ]:
# DRG Ramp down mode (no-dwell low)
#   - Requires IIO backend support for DRCTL toggling
#
# 26.8 GHz __       _____      _____
#             \    |     \    |
#              \   |      \   |
#               \  |       \  |
#                \ |        \ |
# 23.8 GHz __     \|         \|
#             <--100us-->
#
fmcw.digital_ramp_config(
    enable=True,
    mode=ad9910.digital_ramp_generator.mode.RAMP_DOWN,
    freq_min=23.8e9,
    freq_max=26.8e9,
    dec_ramp_time=50e-6,
    toggle_en=1,            # enable auto DRCTL toggling
    ramp_delay=50e-6,       # 50 us delay between ramps
)

In [ ]:
# DRG Normal mode (dwell at both high and low)
#   - Requires IIO backend support for DRCTL toggling
#
# 26.8 GHz __      _____               _____
#                 /     \             /
#                /       \           /
#               /         \         /
#              /           \       /
# 23.8 GHz __ /             \_____/
#             <--100us-->
#
fmcw.digital_ramp_config(
    enable=True,
    mode=ad9910.digital_ramp_generator.mode.BIDIRECTIONAL,
    freq_min=23.8e9,
    freq_max=26.8e9,
    inc_ramp_time=50e-6,
    dec_ramp_time=50e-6,
    toggle_en=1,            # enable auto DRCTL toggling
    ramp_delay=50e-6,       # 50 us delay between ramps
)

In [ ]:
# Disable DRG
fmcw.digital_ramp_config(False)

##### RAM Control

In [ ]:
# Configure RAM Control Profile 0
fmcw.ram_control_profile_config(
    profile=0,
    mode=ad9910.ram_control.mode.RAMP_UP_CONTINUOUS,
    addr_range=(0, 999), # 1000 steps
    rate=10e6 # 1000 / 10 MSPS = 100 us ramp time
)

# loading frequency pattern into RAM:
# 26.8 GHz __      .          .
#                 /|         /|
#                / |        / |
#               /  |       /  |
#              /   |      /   |
# 23.8 GHz __ /    |_____/    |_____
#             <--100us-->
#
freqs = np.linspace(23.8e9, 26.8e9, 500).astype(np.uint64)
freqs = np.concatenate((freqs, np.linspace(23.8e9, 23.8e9, 500).astype(np.uint64)))

# Load frequencies into RAM and enable RAM Mode
fmcw.ram_control_config(
    enable=True,
    frequency_np=freqs
)

In [ ]:
# Disable RAM Mode
fmcw.ram_control_config(False)

##### Debug

In [ ]:
# ADF41513 (PLL) register dump
for reg in reversed(adf41513.reg):
    print(f"{reg.name} = {hex(fmcw.pll.reg_read(reg))}")

In [ ]:
# AD9910 (DDS) register dump
for reg in ad9910.reg:
    print(f"{reg.name} = {hex(fmcw.dds.reg_read(reg))}")

##### Dashboard

In [ ]:
%gui qt
from admfm8000_dashboard import ADMFM8000Dashboard

dash = ADMFM8000Dashboard(fmcw=fmcw)
dash.show()